# 05 - EM-LDDMM Registration

Run the Step 5 EM-LDDMM workflow from the prepared dataset produced by notebook 04. This notebook is a guided front end around `wsi_pipeline.registration`: it resolves the same run plan as the staged CLI, executes the package workflow, and displays the artifacts written under the registration output directory.

Atlas registration is optional. Leave `ATLAS_PATH = None` for atlas-free self-alignment, or set an atlas path plus either `INIT_AFFINE_PATH` or both `ORIENTATION_FROM` and `ORIENTATION_TO`.

## Step 1: Setup and Parameters

In [ ]:
from __future__ import annotations

import json
import shlex
from pathlib import Path

import pandas as pd
from IPython.display import HTML, Image, JSON, display

from wsi_pipeline.registration import plan_emlddmm_workflow, run_emlddmm_workflow
from wsi_pipeline.registration.orientation import (
    list_valid_orientation_codes,
    orientation_validation_rule,
)
from wsi_pipeline.registration.targets import load_manifest

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except NameError:
    pass

In [ ]:
# Docker-friendly defaults. Edit these values for local paths or atlas resources.
DATASET_ROOT = Path("/output/tissue_sections").expanduser()
TARGET_SOURCE = DATASET_ROOT
TARGET_SOURCE_FORMAT = "auto"  # auto, prepared-dir, or precomputed
PRECOMPUTED_MANIFEST = None
REGISTRATION_OUTPUT = DATASET_ROOT / "emlddmm"
EMLDDMM_CONFIG = None
PRESET = "macaque-notebook"
DEVICE = "auto"

# Run behavior. The default executes every enabled stage when inputs are valid.
RUN_REGISTRATION = True
DRY_RUN = False
WRITE_QC_REPORT = True
WRITE_NOTEBOOK_BUNDLE = False

# Optional atlas resources. Atlas mode activates only when ATLAS_PATH is set.
ATLAS_PATH = None  # Example: Path("/data/resources/atlas.vtk")
LABEL_PATH = None  # Example: Path("/data/resources/atlas_labels.vtk")
INIT_AFFINE_PATH = None
ORIENTATION_FROM = None  # Example: "PIR"
ORIENTATION_TO = None  # Example: "RIP"
ATLAS_UNIT_SCALE = None  # Preset default: 1000.0
TARGET_UNIT_SCALE = None  # Preset default: 1.0

# Optional between-slice filling and transformation graph execution.
UPSAMPLE_BETWEEN_SLICES = False
UPSAMPLE_MODE = "seg"  # seg or img
RUN_TRANSFORMATION_GRAPH = False
TRANSFORMATION_GRAPH_SCRIPT = None

In [ ]:
def optional_path(value):
    if value in (None, ""):
        return None
    return Path(value).expanduser()


atlas_path = optional_path(ATLAS_PATH)
label_path = optional_path(LABEL_PATH)
init_affine_path = optional_path(INIT_AFFINE_PATH)
precomputed_manifest = optional_path(PRECOMPUTED_MANIFEST)
emlddmm_config = optional_path(EMLDDMM_CONFIG)
transformation_graph_script = optional_path(TRANSFORMATION_GRAPH_SCRIPT)

if atlas_path is None:
    label_path = None
    init_affine_path = None
    orientation_from = None
    orientation_to = None
else:
    orientation_from = ORIENTATION_FROM.upper() if ORIENTATION_FROM else None
    orientation_to = ORIENTATION_TO.upper() if ORIENTATION_TO else None
    if init_affine_path is None and not (orientation_from and orientation_to):
        raise ValueError(
            "Atlas registration requires INIT_AFFINE_PATH or both "
            "ORIENTATION_FROM and ORIENTATION_TO. No orientation guess is applied."
        )

workflow_kwargs = {
    "dataset_root": DATASET_ROOT,
    "target_source": TARGET_SOURCE,
    "target_source_format": TARGET_SOURCE_FORMAT,
    "registration_output": REGISTRATION_OUTPUT,
    "atlas": atlas_path,
    "label": label_path,
    "emlddmm_config": emlddmm_config,
    "preset": PRESET,
    "init_affine": init_affine_path,
    "orientation_from": orientation_from,
    "orientation_to": orientation_to,
    "atlas_unit_scale": ATLAS_UNIT_SCALE,
    "target_unit_scale": TARGET_UNIT_SCALE,
    "device": DEVICE,
    "precomputed_manifest": precomputed_manifest,
    "upsample_between_slices": UPSAMPLE_BETWEEN_SLICES,
    "upsample_mode": UPSAMPLE_MODE if UPSAMPLE_BETWEEN_SLICES else None,
    "run_transformation_graph": RUN_TRANSFORMATION_GRAPH,
    "transformation_graph_script": transformation_graph_script,
    "write_notebook_bundle": WRITE_NOTEBOOK_BUNDLE,
    "write_qc_report": WRITE_QC_REPORT,
}

print(orientation_validation_rule())
print("Example valid orientation codes:", ", ".join(list_valid_orientation_codes()[:8]), "...")

## Step 2: Validate Step 4 Handoff

In [ ]:
samples_tsv = DATASET_ROOT / "samples.tsv"
manifest_path = precomputed_manifest or (DATASET_ROOT / "emlddmm_dataset_manifest.json")
sidecars = sorted(
    path
    for path in DATASET_ROOT.glob("*.json")
    if path.name != "emlddmm_dataset_manifest.json"
)

missing = [path for path in (samples_tsv, manifest_path) if path is not None and not path.exists()]
if missing:
    raise FileNotFoundError("Missing Step 4 output(s): " + ", ".join(str(path) for path in missing))
if not sidecars:
    raise FileNotFoundError(f"No per-slice JSON sidecars found in {DATASET_ROOT}")

print(f"Dataset root:        {DATASET_ROOT}")
print(f"samples.tsv:         {samples_tsv}")
print(f"Manifest:            {manifest_path}")
print(f"Sidecar JSON files:  {len(sidecars)}")
print(f"Registration output: {REGISTRATION_OUTPUT}")

## Step 3: Preview Target Manifest

In [ ]:
manifest = load_manifest(manifest_path)
entries = list(manifest.get("entries", []))

def is_present(entry):
    return str(entry.get("status", "")).strip().lower() in {"present", "true", "1"}


present_count = sum(1 for entry in entries if is_present(entry))
missing_count = len(entries) - present_count

overview = {
    "space": manifest.get("space"),
    "dv_um": manifest.get("dv_um"),
    "z_axis_um": manifest.get("z_axis_um"),
    "full_grid_count": manifest.get("full_grid_count"),
    "dense_present_count": manifest.get("dense_present_count"),
    "present_count_from_entries": present_count,
    "missing_count_from_entries": missing_count,
    "target_ext": manifest.get("target_ext"),
}
display(JSON(overview, expanded=True))

preview_columns = [
    "grid_index",
    "present_rank",
    "sample_id",
    "status",
    "image_filename",
    "z_position_um",
]
preview_rows = [{column: entry.get(column) for column in preview_columns} for entry in entries[:12]]
display(pd.DataFrame(preview_rows))

## Step 4: Resolve the Step 5 Run Plan

In [ ]:
resolved_plan = plan_emlddmm_workflow(**workflow_kwargs, dry_run=True)

plan_overview = {
    "mode": resolved_plan.mode,
    "backend": resolved_plan.backend_name,
    "target_source": str(resolved_plan.target_source),
    "target_source_format": resolved_plan.target_source_format,
    "registration_output": str(resolved_plan.registration_output),
    "atlas_path": str(resolved_plan.atlas_path) if resolved_plan.atlas_path else None,
    "label_path": str(resolved_plan.label_path) if resolved_plan.label_path else None,
    "device_requested": resolved_plan.device_requested,
    "device_used": resolved_plan.device_used,
    "atlas_init_mode": resolved_plan.atlas_init_mode,
    "enabled_stages": resolved_plan.enabled_stages,
    "skipped_stages": resolved_plan.skipped_stages,
    "working_resolution_um": resolved_plan.working_resolution_um,
    "target_downsampling": resolved_plan.target_downsampling,
    "atlas_downsampling": resolved_plan.atlas_downsampling,
}
display(JSON(plan_overview, expanded=True))
display(JSON(resolved_plan.pre_resampling_plan.model_dump(mode="python"), expanded=True))

if resolved_plan.warnings:
    print("Warnings:")
    for warning in resolved_plan.warnings:
        print(f"- {warning}")

## Step 5: CLI Replay Command

In [ ]:
def append_path_arg(args, flag, value):
    value = optional_path(value)
    if value is not None:
        args.extend([flag, str(value)])


cli_args = [
    "python",
    "scripts/run_pipeline.py",
    "step5",
    "--dataset-root",
    str(DATASET_ROOT),
    "--target-source",
    str(TARGET_SOURCE),
    "--target-source-format",
    TARGET_SOURCE_FORMAT,
    "--registration-output",
    str(REGISTRATION_OUTPUT),
    "--preset",
    PRESET,
]
append_path_arg(cli_args, "--emlddmm-config", emlddmm_config)
append_path_arg(cli_args, "--atlas", atlas_path)
append_path_arg(cli_args, "--label", label_path)
append_path_arg(cli_args, "--init-affine", init_affine_path)
append_path_arg(cli_args, "--precomputed-manifest", precomputed_manifest)
append_path_arg(cli_args, "--transformation-graph-script", transformation_graph_script)

if orientation_from and orientation_to:
    cli_args.extend(["--orientation-from", orientation_from, "--orientation-to", orientation_to])
if ATLAS_UNIT_SCALE is not None:
    cli_args.extend(["--atlas-unit-scale", str(ATLAS_UNIT_SCALE)])
if TARGET_UNIT_SCALE is not None:
    cli_args.extend(["--target-unit-scale", str(TARGET_UNIT_SCALE)])
if DEVICE:
    cli_args.extend(["--device", DEVICE])
if UPSAMPLE_BETWEEN_SLICES:
    cli_args.extend(["--upsample-between-slices", "--upsample-mode", UPSAMPLE_MODE])
if RUN_TRANSFORMATION_GRAPH:
    cli_args.append("--run-transformation-graph")
if WRITE_QC_REPORT:
    cli_args.append("--write-qc-report")
if WRITE_NOTEBOOK_BUNDLE:
    cli_args.append("--write-notebook-bundle")
if DRY_RUN:
    cli_args.append("--dry-run")

line_join = " " + "\\" + "\n  "
print(line_join.join(shlex.quote(part) for part in cli_args))

## Step 6: Run Registration

In [ ]:
if RUN_REGISTRATION:
    result = run_emlddmm_workflow(**workflow_kwargs, dry_run=DRY_RUN)
    print(f"Mode:         {result.mode}")
    print(f"Plan JSON:    {result.plan_path}")
    print(f"Summary JSON: {result.summary_path}")
    print(f"Log file:     {result.log_path}")
    if result.provenance_path is not None:
        print(f"Provenance:   {result.provenance_path}")
    if result.reproduce_command_path is not None:
        print(f"Replay cmd:   {result.reproduce_command_path}")
    if result.report_path is not None:
        print(f"QC report:    {result.report_path}")
else:
    result = None
    print("RUN_REGISTRATION is False; resolved_plan is available for inspection.")

## Step 7: Review Summary and Artifacts

In [ ]:
summary_path = result.summary_path if result is not None else REGISTRATION_OUTPUT / "registration_summary.json"
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    summary_overview = {
        key: summary.get(key)
        for key in (
            "mode",
            "backend_name",
            "device_used",
            "atlas_init_mode",
            "enabled_stages",
            "completed_stages",
            "skipped_stages",
            "warnings",
        )
    }
    display(JSON(summary_overview, expanded=True))
    display(pd.DataFrame(summary.get("stage_timeline", [])))
else:
    print(f"No summary found yet: {summary_path}")

In [ ]:
report_path = REGISTRATION_OUTPUT / "registration_report.html"
report_manifest_path = REGISTRATION_OUTPUT / "registration_report.json"
replay_path = REGISTRATION_OUTPUT / "reproduce_step5_command.txt"
log_path = REGISTRATION_OUTPUT / "registration.log"

if report_path.exists():
    display(HTML(f'<p><strong>QC report:</strong> <a href="{report_path}" target="_blank">{report_path}</a></p>'))
if replay_path.exists():
    print("Replay command:")
    print(replay_path.read_text(encoding="utf-8"))
if log_path.exists():
    print("Last 60 log lines:")
    print("\n".join(log_path.read_text(encoding="utf-8").splitlines()[-60:]))

In [ ]:
if report_manifest_path.exists():
    report_manifest = json.loads(report_manifest_path.read_text(encoding="utf-8"))
    for stage in report_manifest.get("stages", []):
        gallery = stage.get("gallery", {})
        images = gallery.get("images", [])[:4]
        print(f"{stage.get('name')}: {stage.get('status')} ({gallery.get('selected_count', 0)} report image(s))")
        for image in images:
            image_path = REGISTRATION_OUTPUT / image["path"]
            if image_path.exists():
                display(Image(filename=str(image_path), width=520))
else:
    print(f"No report manifest found yet: {report_manifest_path}")

## Output Layout

Typical Step 5 outputs are written under `REGISTRATION_OUTPUT`:

- `resolved_run_plan.json`
- `registration_summary.json`
- `registration.log`
- `run_provenance.json`
- `reproduce_step5_command.txt`
- `self_alignment/`
- optional `atlas_registration/`
- optional `upsampling/`
- optional `registration_report.html` and `registration_report.json`